# LSTM con atencion (direccion del precio)

Modelo LSTM con atencion aditiva sobre los pasos temporales.
Usa la misma tarea: predecir si el precio sube al dia siguiente.

- Raw price only como input.
- Split temporal por mercado.
- StandardScaler ajustado solo en train.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
# Walk up until repo root (contains src/)
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from src.features import build_labels, temporal_split

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Configuracion

- `MARKET_FILES`: por defecto todos los data/raw/prices_*.csv.
- Split 70/30 por mercado, sin mezclar ventanas entre slugs.
- Raw price only (una sola columna de features).

In [ ]:
RAW_DIR = ROOT / "data" / "raw"
MARKET_FILES = sorted(RAW_DIR.glob("prices_*.csv"))

MAX_MARKETS = None
if MAX_MARKETS is not None:
    MARKET_FILES = MARKET_FILES[:MAX_MARKETS]

GROUP_COL = "slug"  # columna que identifica el mercado

LOOKBACK = 30   
TRAIN_RATIO = 0.70
EPOCHS = 80
BATCH_SIZE = 16
HIDDEN_SIZE = 32
LSTM_LAYERS = 2
LEARNING_RATE = 1e-3

SAVE_MODEL = True
MODEL_PATH = PROC_DIR / "lstm_attention.pt"

## 1. Cargar datos y labels

In [4]:
def load_one_market(path: Path) -> pd.DataFrame:
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["date"], utc=True)
    d = d.sort_values("date").reset_index(drop=True)
    d = build_labels(d, price_col="price")
    d = d.dropna().reset_index(drop=True)
    d = d.dropna(subset=["label"]).reset_index(drop=True)
    d["label"] = d["label"].astype(np.float64)
    return d


if not MARKET_FILES:
    raise FileNotFoundError(
        f"No hay CSV en {RAW_DIR}/prices_*.csv. Descarga datos o ajusta MARKET_FILES."
    )

parts = [load_one_market(p) for p in MARKET_FILES]
df = pd.concat(parts, ignore_index=True)

EXCLUDE = {"date", "slug", "question", "label"}
FEATURE_COLS = ["price"]

n_m = df[GROUP_COL].nunique()
print(f"Archivos: {len(MARKET_FILES)}, mercados (slugs): {n_m}, filas totales: {len(df)}")
df.head()

Archivos: 500, mercados (slugs): 500, filas totales: 30559


,date,price,slug,question,label
0,2023-12-29 00:00:00+00:00,0.235,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.0
1,2023-12-30 00:00:00+00:00,0.200,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.0
2,2023-12-31 00:00:00+00:00,0.160,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.0
3,2024-01-01 00:00:00+00:00,0.125,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.0
4,2024-01-02 00:00:00+00:00,0.115,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,1.0


## 2. Split temporal por mercado y escalado (solo train)

In [5]:
train_parts, test_parts = [], []
for _, g in df.groupby(GROUP_COL):
    g = g.sort_values("date").reset_index(drop=True)
    tr, te = temporal_split(g, train_ratio=TRAIN_RATIO, date_col="date")
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].values)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [6]:
def make_sequences(features: np.ndarray, labels: np.ndarray, lookback: int):
    """features: (n, n_feat), labels: (n,) — etiqueta al ultimo dia de la ventana."""
    X_list, y_list = [], []
    for t in range(lookback - 1, len(features)):
        X_list.append(features[t - lookback + 1 : t + 1])
        y_list.append(labels[t])
    return np.stack(X_list, axis=0), np.array(y_list, dtype=np.float32)


def sequences_from_grouped(split_df: pd.DataFrame, lookback: int):
    """Una ventana LSTM nunca cruza dos mercados distintos."""
    X_blocks, y_blocks = [], []
    for _, g in split_df.groupby(GROUP_COL):
        g = g.sort_values("date")
        if len(g) < lookback:
            continue
        Xs = scaler.transform(g[FEATURE_COLS].values)
        ys = g["label"].values.astype(np.float32)
        X_seq, y_seq = make_sequences(Xs, ys, lookback)
        if len(y_seq) == 0:
            continue
        X_blocks.append(X_seq)
        y_blocks.append(y_seq)
    if not X_blocks:
        return np.empty((0, lookback, len(FEATURE_COLS))), np.array([], dtype=np.float32)
    return np.concatenate(X_blocks, axis=0), np.concatenate(y_blocks, axis=0)


X_train, y_train = sequences_from_grouped(train_df, LOOKBACK)
X_test, y_test = sequences_from_grouped(test_df, LOOKBACK)

print(f"Train secuencias: {len(y_train)}, Test secuencias: {len(y_test)}")
if len(y_train) == 0 or len(y_test) == 0:
    raise ValueError(
        "Muy pocas filas por mercado o LOOKBACK demasiado grande. "
        "Anade mas CSV, reduce LOOKBACK o revisa mercados cortos."
    )

Train secuencias: 10512, Test secuencias: 2074


## 3. DataLoaders

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float(),
)

train_loader = DataLoader(
    train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True, drop_last=False
)
test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

## 4. Modelo: LSTM + atencion aditiva

In [8]:
n_features = X_train.shape[2]


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, h: torch.Tensor):
        # h: (batch, time, hidden)
        scores = self.v(torch.tanh(self.W(h))).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(h * weights.unsqueeze(-1), dim=1)
        return context, weights


class LSTMAttentionClassifier(nn.Module):
    def __init__(self, n_feat: int, hidden: int, n_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_feat,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
        )
        self.attn = AdditiveAttention(hidden)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor):
        out, _ = self.lstm(x)
        context, weights = self.attn(out)
        logits = self.fc(context).squeeze(-1)
        return logits, weights


model = LSTMAttentionClassifier(n_features, HIDDEN_SIZE, LSTM_LAYERS).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

LSTMAttentionClassifier(
  (lstm): LSTM(1, 32, num_layers=2, batch_first=True)
  (attn): AdditiveAttention(
    (W): Linear(in_features=32, out_features=32, bias=True)
    (v): Linear(in_features=32, out_features=1, bias=False)
  )
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

## 5. Entrenamiento

In [9]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg = total_loss / len(train_ds)
    if epoch == 1 or epoch % max(1, EPOCHS // 10) == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:4d}  loss {avg:.4f}")

epoch    1  loss 0.5013
epoch    8  loss 0.4943
epoch   16  loss 0.4938
epoch   24  loss 0.4926
epoch   32  loss 0.4931
epoch   40  loss 0.4932
epoch   48  loss 0.4927
epoch   56  loss 0.4914
epoch   64  loss 0.4888
epoch   72  loss 0.4878
epoch   80  loss 0.4872


## 6. Evaluacion en test

Umbral 0.5 en probabilidad (sigmoid(logits)).

In [10]:
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits, attn_w = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_true = yb.numpy()

y_pred = (probs >= 0.5).astype(int)
y_true_int = y_true.astype(int)

acc = accuracy_score(y_true_int, y_pred)
f1 = f1_score(y_true_int, y_pred, zero_division=0)

print(f"Accuracy: {acc:.3f}")
print(f"F1 (binary): {f1:.3f}")

if SAVE_MODEL:
    torch.save(
        {
            "model_state": model.state_dict(),
            "feature_cols": FEATURE_COLS,
            "lookback": LOOKBACK,
            "hidden_size": HIDDEN_SIZE,
            "n_layers": LSTM_LAYERS,
            "scaler_mean": scaler.mean_,
            "scaler_scale": scaler.scale_,
            "group_col": GROUP_COL,
            "n_markets": int(df[GROUP_COL].nunique()),
            "csv_stems": [p.stem for p in MARKET_FILES],
        },
        MODEL_PATH,
    )
    print(f"Guardado: {MODEL_PATH}")

Accuracy: 0.749
F1 (binary): 0.000
Guardado: C:\Users\oscar\repos\predikt\data\processed\lstm_attention.pt
